In [0]:
# ============================================================
# Bronze — Source 17: GA4 Export
#
# Incremental load using watermark pattern.
#
# Source: s3://ecommerce-lakehouse-467091806172-raw-01/source=17_ga4/
# Target: bronze.ga4 catalog in Unity Catalog
#
# Tables:
#   - bronze.ga4.events
#
# Raw format: JSON (daily partitioned)
# ============================================================

import sys
sys.path.append("/Workspace/Users/sutharripal26@gmail.com/ecommerce-lakehouse/pipelines/bronze/shared")
from bronze_utils import get_watermark, update_watermark
from pyspark.sql.functions import col, lit, max as spark_max

spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")

RAW_BUCKET = "s3://ecommerce-lakehouse-467091806172-raw-01"
SOURCE = "17_ga4"

dbutils.fs.ls(f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=18/")

In [0]:
# Cell 2 — Read GA4 events JSON files
watermark = get_watermark(spark, SOURCE)
print(f"Loading files modified after: {watermark}")

path = f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=*/ga4_events_*.json"

df = spark.read.format("json") \
    .load(path) \
    .filter(col("_metadata.file_modification_time") > lit(watermark))

print(f"Raw rows: {df.count()}")
df.printSchema()

In [0]:
# Cell 3 — Flatten GA4 events with correct schema
from pyspark.sql.functions import explode

events_flat = df \
    .select(explode(col("events")).alias("e")) \
    .select(
        col("e.event_name"),
        col("e.event_date"),
        col("e.event_timestamp"),
        col("e.event_value_in_usd"),
        col("e.event_bundle_sequence_id"),
        col("e.platform"),
        col("e.stream_id"),
        col("e.user_id"),
        col("e.user_pseudo_id"),
        col("e.device.browser").alias("device_browser"),
        col("e.device.category").alias("device_category"),
        col("e.device.language").alias("device_language"),
        col("e.device.operating_system").alias("device_os"),
        col("e.geo.city").alias("geo_city"),
        col("e.geo.country").alias("geo_country"),
        col("e.traffic_source.medium").alias("traffic_medium"),
        col("e.traffic_source.source").alias("traffic_source"),
        col("e.user_ltv.revenue").alias("user_ltv_revenue"),
        col("e.user_ltv.currency").alias("user_ltv_currency"),
        col("e.event_params"),
    )

print(f"Flattened events: {events_flat.count()} rows")
events_flat.printSchema()

In [0]:
# Cell 4 — Write to Bronze and update watermark
# Note: event_params kept as array — Silver will extract specific keys
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze.ga4")

events_flat.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("bronze.ga4.events")

latest_ts = df.select(spark_max("_metadata.file_modification_time")).collect()[0][0]
update_watermark(spark, SOURCE, latest_ts, events_flat.count())
print(f"✅ bronze.ga4.events: {events_flat.count()} rows")

In [0]:
count = spark.sql("SELECT COUNT(*) as cnt FROM bronze.ga4.events").collect()[0]['cnt']
print(f"bronze.ga4.events: {count} rows")
spark.sql("SELECT * FROM bronze.pipeline.watermarks WHERE source = '17_ga4'").show()